# Lecture 14: Advanced Transformer Topics
### NLP Course 2027

---

## Learning Outcomes
- Understand cross-lingual and multilingual transformer models
- Apply zero-shot cross-lingual transfer
- Know efficiency techniques: distillation, quantization, LoRA
- Select the right model/strategy for production constraints

**Primary Reference:** *NLP with Transformers* Ch.4 (multilingual) & Ch.8 (efficient transformers)

## 1. Domain Adaptation

Pretrained models learn general language, but domain text differs significantly:

| Domain | Special Vocabulary | Challenge |
|--------|--------------------|-----------|
| Biomedical | COPD, myocardial infarction | Medical abbreviations |
| Legal | tort, plaintiff, habeas corpus | Latin terms, jargon |
| Financial | EBITDA, P/E ratio, yield curve | Numerical expressions |
| Social media | lol, brb, @mentions | Informal, noisy |

### Domain-specific BERT models
| Model | Domain | Trained on |
|-------|--------|------------|
| BioBERT | Biomedical | PubMed + PMC (18B tokens) |
| LegalBERT | Legal | 12GB legal text |
| FinBERT | Finance | Financial news, SEC filings |
| SciBERT | Scientific | Semantic Scholar papers |
| ClinicalBERT | Clinical notes | MIMIC-III |

*Domain models consistently outperform general BERT on in-domain tasks.*

In [1]:
from transformers import pipeline

# Compare general BERT vs BioBERT on biomedical text
# General BERT fill-mask
general_fill = pipeline('fill-mask', model='bert-base-uncased')

# Masking a medical term
masked = 'The patient was diagnosed with acute [MASK] infarction.'
print('General BERT predictions:')
for r in general_fill(masked, top_k=5):
    print(f'  [{r["score"]:.4f}] {r["sequence"]}')

print('\nBioBERT would predict: myocardial (domain-appropriate term)')
print('Usage: pipeline("fill-mask", model="dmis-lab/biobert-base-cased-v1.1")')

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you

General BERT predictions:
  [0.5062] the patient was diagnosed with acute pulmonary infarction.
  [0.0763] the patient was diagnosed with acute lung infarction.
  [0.0617] the patient was diagnosed with acute heart infarction.
  [0.0569] the patient was diagnosed with acute cardiac infarction.
  [0.0395] the patient was diagnosed with acute kidney infarction.

BioBERT would predict: myocardial (domain-appropriate term)
Usage: pipeline("fill-mask", model="dmis-lab/biobert-base-cased-v1.1")


## 2. Cross-Lingual and Multilingual Models

### Why Cross-Lingual NLP?
- Train on labeled data in one language (e.g., English)
- Apply to other languages without any translated labels
- Critical for low-resource languages (Swahili, Vietnamese, etc.)

### Key Models
| Model | Languages | Training | Strength |
|-------|-----------|----------|---------|
| mBERT | 104 | Multilingual Wikipedia | Good baseline |
| XLM-RoBERTa | 100 | CC-100 (2.5TB) | State-of-the-art |
| mT5 | 101 | mC4 (>1TB) | Generation tasks |
| NLLB | 200 | Meta's NLLB corpus | Translation |

### Zero-Shot Cross-Lingual Transfer
```
Train:  English labeled data  ->  English model
Test:   German text  -> same model  ->  German predictions
        (no German labels used!)
```

In [2]:
from transformers import pipeline

# XLM-RoBERTa for cross-lingual sentiment
classifier = pipeline(
    'text-classification',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment'
)

multi_texts = [
    ('English', 'This NLP course is absolutely amazing!'),
    ('Spanish', 'Este curso de PLN es absolutamente increible!'),
    ('French', 'Ce cours de NLP est absolument incroyable!'),
    ('German', 'Dieser NLP-Kurs ist absolut erstaunlich!'),
    ('Chinese', 'zhe ge NLP ke cheng fei chang jing cai!'),
]

print('Cross-lingual sentiment classification:')
print('-' * 70)
for lang, text in multi_texts:
    result = classifier(text)[0]
    print(f'{lang:12s} | {result["label"]:10s} ({result["score"]:.3f}) | {text[:40]}')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Cross-lingual sentiment classification:
----------------------------------------------------------------------
English      | positive   (0.938) | This NLP course is absolutely amazing!
Spanish      | positive   (0.939) | Este curso de PLN es absolutamente incre
French       | positive   (0.923) | Ce cours de NLP est absolument incroyabl
German       | positive   (0.920) | Dieser NLP-Kurs ist absolut erstaunlich!
Chinese      | neutral    (0.395) | zhe ge NLP ke cheng fei chang jing cai!


In [3]:
# Multilingual NER
ner_pipe = pipeline('ner',
                    model='Babelscape/wikineural-multilingual-ner',
                    aggregation_strategy='simple')

texts = [
    'Barack Obama was born in Hawaii and became the 44th President.',
    'Emmanuel Macron est le president de la France depuis 2017.',
]
for text in texts:
    entities = ner_pipe(text)
    print(f'Text: {text[:60]}...')
    for ent in entities:
        print(f'  [{ent["entity_group"]:6s}] {ent["word"]}')
    print()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: Barack Obama was born in Hawaii and became the 44th Presiden...
  [PER   ] Barack Obama
  [LOC   ] Hawaii

Text: Emmanuel Macron est le president de la France depuis 2017....
  [PER   ] Emmanuel Macron
  [LOC   ] France



## 3. Efficient Transformers

BERT-base has 110M parameters. GPT-3 has 175B. Production deployment requires efficiency.

```
Model compression techniques:

  Knowledge Distillation   Teacher -> Student (66M -> 40% faster)
  Quantization             FP32 -> INT8 (4x smaller, 2-4x faster)
  Pruning                  Remove small weights (50-90% sparsity)
  Parameter-efficient FT   Train only 0.1-1% params (LoRA)
```

### LoRA (Low-Rank Adaptation)
```
Original weight: W (d x k)
LoRA adds:       dW = A x B  where A:(d x r), B:(r x k), r << k

Only A and B are trained. r=8 means ~0.1% of parameters!
    W_output = W_frozen + (A x B) * scaling
```

In [4]:
from transformers import AutoModelForSequenceClassification
import torch

# Compare BERT-base vs DistilBERT vs ALBERT
models = {
    'BERT-base':    'bert-base-uncased',
    'DistilBERT':   'distilbert-base-uncased',
    'ALBERT-base':  'albert-base-v2',
}
print(f'{"Model":20s} {"Parameters":>15s} {"Size (MB)":>12s}')
print('-' * 50)
for name, model_name in models.items():
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        params = sum(p.numel() for p in model.parameters())
        size_mb = params * 4 / 1e6
        print(f'{name:20s} {params:>15,} {size_mb:>11.0f}MB')
    except Exception as e:
        print(f'{name:20s}  error: {e}')

Model                     Parameters    Size (MB)
--------------------------------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT-base                109,483,778         438MB
DistilBERT                66,955,010         268MB


Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ALBERT-base               11,685,122          47MB


In [5]:
import torch
import time
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Inference speed: BERT vs DistilBERT
texts = [
    'The transformer architecture revolutionized NLP.',
    'Fine-tuning pretrained models is efficient.',
] * 15  # 30 texts

def benchmark_model(model_name, texts, n=3):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.eval()
    encoded = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    times = []
    with torch.no_grad():
        for _ in range(n):
            t0 = time.time()
            _ = model(**encoded)
            times.append(time.time() - t0)
    return sum(times) / n

bert_t = benchmark_model('bert-base-uncased', texts)
distil_t = benchmark_model('distilbert-base-uncased', texts)
print(f'BERT-base:   {bert_t*1000:.1f}ms')
print(f'DistilBERT:  {distil_t*1000:.1f}ms')
print(f'Speedup:     {bert_t/distil_t:.1f}x')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT-base:   473.2ms
DistilBERT:  243.7ms
Speedup:     1.9x


In [6]:
# LoRA with PEFT library
# !pip install peft

try:
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import AutoModelForSequenceClassification

    base_model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=2)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=['query', 'value'],
    )
    lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters()

except ImportError:
    print('Install PEFT: pip install peft')
    print('LoRA reduces trainable params from 110M to ~300K (0.3%)')
    print('Achieves 95-99% of full fine-tuning performance')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [7]:
# Quantization: INT8 inference on CPU
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f'FP32 model params: {params:,}')
print(f'FP32 model size:   {params*4/1e6:.1f} MB')

# Dynamic quantization
quantized = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8)
print('INT8 quantized model ready.')
print('Benefits: ~2-4x smaller on disk, faster on CPU, minimal accuracy loss')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FP32 model params: 66,955,010
FP32 model size:   267.8 MB
INT8 quantized model ready.
Benefits: ~2-4x smaller on disk, faster on CPU, minimal accuracy loss


In [8]:
# Sliding window for long documents (BERT max = 512 tokens)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

def sliding_window_classify(text, model, tokenizer, window=512, stride=256):
    """Classify long documents via overlapping chunks."""
    token_ids = tokenizer(text, add_special_tokens=False)['input_ids']
    all_logits = []
    for start in range(0, len(token_ids), stride):
        end = start + window - 2  # space for [CLS] + [SEP]
        chunk = [tokenizer.cls_token_id] + token_ids[start:end] + [tokenizer.sep_token_id]
        inp = torch.tensor([chunk])
        with torch.no_grad():
            logits = model(inp).logits
        all_logits.append(logits)
        if end >= len(token_ids):
            break
    avg_logits = torch.stack(all_logits).mean(0)
    return avg_logits.argmax(-1).item()

print('Sliding window approach for documents > 512 tokens.')
print('Chunk -> classify -> average logits -> final prediction.')

Sliding window approach for documents > 512 tokens.
Chunk -> classify -> average logits -> final prediction.


## Practice Exercises

See **`Lecture-14-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Need | Solution | Model/Technique |
|------|----------|----------------|
| Domain text | Domain fine-tuning | BioBERT, LegalBERT |
| Multiple languages | Multilingual model | XLM-RoBERTa |
| Small/fast model | Distillation | DistilBERT |
| Minimal GPU | Quantization | INT8, mixed precision |
| Limited labels | PEFT / LoRA | 0.1% trainable params |
| Long documents | Sliding window | Chunk + aggregate |

**Next Lecture**: Dialogue & Chatbot Systems.

---
*Book reference: NLP with Transformers Ch.4, 8*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**